In [ ]:
#CREACION DE CARPETAS
!mkdir -p datos/crudos
!mkdir -p datos/procesados
!mkdir -p datos/externos
print("Carpetas creadas !!!!!!")

In [ ]:
#IMPORTACION DE LIBRERIAS
import hashlib
from pathlib import Path

import pandas as pd
print("Libreria importada con exito!!")

In [ ]:
#Rutas de trabajo ya definidas "atajos"
BASE_DIR = Path("/content/")
DATA_RAW = BASE_DIR/"datos"/"crudos"
DATA_EXTERNAL = BASE_DIR/"datos"/"externos"
DATA_PROCESSED = BASE_DIR/"datos"/"procesados"
print(f"Datos crudos-> {DATA_RAW}")
print(f"Datos procesados-> {DATA_PROCESSED}")
print(f"Datos externos -> {DATA_EXTERNAL}")
DOCUMENTOS = BASE_DIR/"datos"/"documentos"
DOCUMENTOS.mkdir(parents=True, exist_ok=True)
print(f"Carpetas creadas con exito en {DOCUMENTOS}")

In [ ]:
df_limpio = pd.read_csv(DATA_PROCESSED / "dataset_limpio.csv")

In [ ]:
# PREPARAR GOBERNANZA DE DATOS
ruta_gobernanza = Path("/content/datos/gobernanza")
ruta_documentos = ruta_gobernanza / "documentos"

ruta_gobernanza.mkdir(parents=True, exist_ok=True)
ruta_documentos.mkdir(parents=True, exist_ok=True)

print("Carpetas de gobernanza creadas correctamente")

In [ ]:
# MATRIZ DE CLASIFICACION DE DATOS

matriz_clasificacion = pd.DataFrame([
    ["id", "Identificador", "Interno", "Identificar cada pedido", "Control de acceso"],
    ["id_repartidor", "Identificador", "Interno", "Identificar al repartidor", "Control de acceso"],
    ["edad_repartidor", "Numerico", "Interno", "Analizar caracteristicas del repartidor", "Minimizacion"],
    ["calificacion_repartidor", "Numerico", "Interno", "Analizar desempeño del repartidor", "Minimizacion"],
    ["fecha_pedido", "Fecha", "Interno", "Analizar comportamiento temporal", "Control de acceso"],
    ["hora_pedido", "Hora", "Interno", "Analizar horarios de pedidos", "Minimizacion"],
    ["hora_recogida_pedido", "Hora", "Interno", "Analizar tiempos de recogida", "Minimizacion"],
    ["condiciones_climaticas", "Categorico", "Interno", "Analizar condiciones del entorno", "Control de acceso"],
    ["densidad_trafico", "Categorico", "Interno", "Analizar condiciones del trafico", "Control de acceso"],
    ["tipo_pedido", "Categorico", "Interno", "Analizar tipos de pedido", "Control de acceso"],
    ["festival", "Categorico", "Interno", "Analizar efecto de festividades", "Control de acceso"],
    ["ciudad", "Categorico", "Interno", "Analizar ubicacion general", "Minimizacion"],
    ["tiempo_entrega_min", "Numerico", "Interno", "Analizar tiempos de entrega", "Control de acceso"],
    ["latitud_restaurante", "Geografico", "Interno", "Analizar ubicacion del restaurante", "Minimizacion"],
    ["longitud_restaurante", "Geografico", "Interno", "Analizar ubicacion del restaurante", "Minimizacion"],
    ["latitud_ubicacion_entrega", "Geografico", "Interno", "Analizar ubicacion de entrega", "Minimizacion"],
    ["longitud_ubicacion_entrega", "Geografico", "Interno", "Analizar ubicacion de entrega", "Minimizacion"],
    ["condicion_vehiculo", "Numerico", "Interno", "Analizar condicion del vehiculo", "Control de acceso"],
    ["tipo_vehiculo", "Categorico", "Interno", "Analizar tipo de vehiculo", "Control de acceso"],
    ["multiples_entregas", "Numerico", "Interno", "Analizar cantidad de entregas", "Control de acceso"]
], columns=[
    "campo",
    "tipo",
    "clasificacion",
    "proposito",
    "control"
])

matriz_clasificacion

In [ ]:
# GUARDAR MATRIZ DE CLASIFICACION

ruta_matriz = ruta_documentos / "matriz_clasificacion.xlsx"

matriz_clasificacion.to_excel(
    ruta_matriz,
    index=False
)

print("Matriz de clasificacion guardada correctamente")

In [ ]:
# PSEUDONIMIZACION DEL ID DEL REPARTIDOR
SALT = "ProyectoIDS222"  # Texto que modifica el hash

def pseudonimizar(valor):
    """Convierte el ID del repartidor en un código único de 12 caracteres"""
    texto = f"{SALT}|{valor}".encode('utf-8')
    return hashlib.sha256(texto).hexdigest()[:12]

df_gobernanza = df_limpio.copy()

df_gobernanza["id_repartidor_original"] = df_gobernanza["id_repartidor"]

df_gobernanza["id_repartidor"] = (
    df_gobernanza["id_repartidor"]
    .astype(str)
    .apply(pseudonimizar)
)

print("Pseudonimización aplicada correctamente:")
print(df_gobernanza[["id_repartidor_original", "id_repartidor"]].head())

In [ ]:
# CREAR VISTA ANALITICA PROTEGIDA

df_protegido = df_gobernanza.drop(
    columns=["id_repartidor_original"]
)

print("VISTA ANALITICA PROTEGIDA")
df_protegido[
    ["id", "id_repartidor", "edad_repartidor", "tipo_vehiculo"]
].head()

In [ ]:
# GENERALIZAR EDAD

def generalizar_edad(edad):
    if edad < 20:
        return "Menor de 20"
    elif edad < 30:
        return "20-29"
    elif edad < 40:
        return "30-39"
    elif edad < 50:
        return "40-49"
    else:
        return "50 o mas"

df_protegido["rango_edad"] = df_protegido["edad_repartidor"].apply(
    generalizar_edad
)

print("Generalizacion de edad aplicada")

df_protegido[
    ["edad_repartidor", "rango_edad"]
].head(10)

In [ ]:
# GUARDAR VISTA ANALITICA PROTEGIDA

ruta_protegido = ruta_gobernanza / "dataset_protegido.csv"

df_protegido.to_csv(
    ruta_protegido,
    index=False
)

print("Vista analitica protegida guardada correctamente")

In [ ]:
# VALIDAR EXPOSICION DE IDENTIFICADORES

columnas_prohibidas = [
    "id_repartidor_original"
]

expuestos = [
    columna
    for columna in columnas_prohibidas
    if columna in df_protegido.columns
]

if len(expuestos) == 0:
    print("VALIDACION CORRECTA: no se encontraron identificadores originales")
else:
    print("ADVERTENCIA: se encontraron identificadores originales")
    print(expuestos)

In [ ]:
# MATRIZ DE RIESGOS

matriz_riesgos = pd.DataFrame([
    ["Exposicion del ID del repartidor", "Media", "Alta",
     "Pseudonimizar el identificador", "Responsable de datos",
     "Vista protegida", 6, "Alto"],

    ["Exposicion de ubicaciones", "Media", "Alta",
     "Minimizar y controlar el acceso a datos geograficos", "Responsable de datos",
     "Vista protegida", 6, "Alto"],

    ["Acceso no autorizado", "Media", "Alta",
     "Aplicar controles de acceso", "Administrador",
     "Control de acceso", 6, "Alto"],

    ["Uso de datos innecesarios", "Media", "Media",
     "Aplicar minimizacion de datos", "Responsable de datos",
     "Matriz de clasificacion", 4, "Medio"],

    ["Exposicion de datos de repartidores", "Baja", "Alta",
     "Limitar el acceso a los datos de repartidores", "Responsable de datos",
     "Matriz de acceso", 3, "Medio"],

    ["Conservacion de datos por tiempo innecesario", "Media", "Media",
     "Establecer reglas de retencion de datos", "Responsable de datos",
     "Politica de gobernanza", 4, "Medio"]
], columns=[
    "riesgo",
    "probabilidad",
    "impacto",
    "control",
    "responsable",
    "evidencia",
    "puntaje",
    "nivel"
])

matriz_riesgos

In [ ]:
# GUARDAR MATRIZ DE RIESGOS

ruta_riesgos = ruta_documentos / "matriz_riesgos.xlsx"

matriz_riesgos.to_excel(
    ruta_riesgos,
    index=False
)

print("Matriz de riesgos guardada correctamente")

In [ ]:
# POLITICA DE GOBERNANZA DE DATOS

politica = """
# Politica de Gobernanza de Datos

## Objetivo
Establecer controles para el uso responsable de los datos del proyecto de logistica.

## Clasificacion
Los datos se clasifican segun su tipo, proposito y nivel de acceso.

## Proteccion
- Los identificadores de repartidores se pseudonimizan.
- Las edades se generalizan en rangos.
- Los datos geograficos deben manejarse con minimizacion y control de acceso.
- La vista analitica no debe contener identificadores originales.

## Acceso
El acceso a los datos debe limitarse segun la necesidad de uso.

## Validacion
Se realizan controles para detectar la exposicion de identificadores originales.

## Responsabilidades
Los responsables de datos deben mantener los controles y revisar los riesgos documentados.
"""

ruta_politica = ruta_documentos / "politica_gobernanza.md"

ruta_politica.write_text(
    politica,
    encoding="utf-8"
)

print("Politica de gobernanza guardada correctamente")

In [ ]:
# MATRIZ DE ACCESO

matriz_acceso = pd.DataFrame([
    ["Administrador", "Todos los datos", "Lectura y escritura",
     "Gestionar y controlar el acceso a los datos"],

    ["Responsable de datos", "Datos de análisis y gobernanza", "Lectura y escritura",
     "Supervisar calidad, protección y gobernanza"],

    ["Analista", "Vista analitica protegida", "Lectura",
     "Realizar análisis sin acceder a identificadores originales"],

    ["Usuario de consulta", "Datos agregados", "Lectura",
     "Consultar resultados del análisis"]
], columns=[
    "rol",
    "datos_permitidos",
    "nivel_acceso",
    "proposito"
])

matriz_acceso

In [ ]:
# GUARDAR MATRIZ DE ACCESO
ruta_acceso = ruta_documentos / "matriz_acceso.xlsx"
matriz_acceso.to_excel(ruta_acceso, index=False)
print("Matriz de acceso guardada correctamente")

In [ ]:
# REGLAS DE RETENCION DE DATOS

reglas_retencion = pd.DataFrame([
    ["Datos de pedidos", "Mientras sean necesarios para el analisis", "Eliminar cuando ya no sean necesarios"],
    ["Datos de repartidores", "Mientras sean necesarios para el analisis", "Eliminar o anonimizar"],
    ["Datos geograficos", "Mientras sean necesarios para el analisis", "Eliminar cuando ya no sean necesarios"],
    ["Datos pseudonimizados", "Mientras sean necesarios para el proyecto", "Eliminar al finalizar su uso"]
], columns=[
    "tipo_dato",
    "periodo_retencion",
    "accion_al_finalizar"
])

reglas_retencion

In [ ]:
# REFLEXION ETICA

reflexion_etica = """
El proyecto debe utilizar los datos de forma responsable, evitando exponer
informacion que permita identificar directamente a los repartidores.

La pseudonimizacion de los identificadores y la generalizacion de la edad
permiten reducir la exposicion de informacion personal durante el analisis.

Los datos de ubicacion requieren especial cuidado, ya que pueden proporcionar
informacion sobre lugares relacionados con los pedidos y las entregas.

El acceso a los datos debe limitarse segun la funcion de cada usuario y los
datos deben conservarse solamente durante el tiempo necesario para el proyecto.

Estas medidas buscan mantener la utilidad de los datos para el analisis
logistico sin utilizar informacion personal de manera innecesaria.
"""

print(reflexion_etica)

In [ ]:
# GUARDAR REGLAS DE RETENCION

ruta_retencion = ruta_documentos / "reglas_retencion.xlsx"

reglas_retencion.to_excel(
    ruta_retencion,
    index=False
)

print("Reglas de retencion guardadas correctamente")

In [ ]:
# ARCHIVOS GENERADOS

print("ARCHIVOS GENERADOS:")

for archivo in ruta_documentos.iterdir():
    print("-", archivo.name)